# YouTube Spam Collection — End-to-End Machine Learning Workflow

This notebook demonstrates a concise end-to-end Scikit-learn classification workflow using the UCI YouTube Spam Collection dataset (dataset ID **380**): 1,956 real YouTube comments labelled as spam (`1`) or legitimate/ham (`0`).

**Source:** [UCI Machine Learning Repository — YouTube Spam Collection](https://archive.ics.uci.edu/dataset/380/youtube+spam+collection)

**Requirements:** `pip install ucimlrepo scikit-learn pandas`

## Step 1 — Data Loading

In [1]:
pip install ucimlrepo scikit-learn panda

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 66.6 kB/s  0:02:01a 0:00:03m eta 0:00:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 79.9 kB/s  0:04:40 eta 0:00:010:00:08
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
  Created wheel for panda: filename=panda-0.3.1-py3-none-any.whl size=7286 sha256=4b7660951b80c8689d80fad13d632ed6e61a34d07b96ec573c6bf826a8cb9dc4
  Stored in directory: /Users/khushgopani/Library/Caches/pip/wheels/b3/62/ac/41e27178e8597be2da908681ca1e75e66255a488040cca1638
Successfully built panda
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [scikit-learn]m━━━━ 8/9 [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [12]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

# Fetch dataset directly by UCI dataset ID
youtube_spam = fetch_ucirepo(id=380)

# Separate features and target
X = youtube_spam.data.features.copy()
y = youtube_spam.data.targets.copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFirst five feature rows:")

X.head()

Features shape: (1956, 3)
Target shape: (1956, 1)

First five feature rows:


,AUTHOR,DATE,CONTENT
0,Julius NM,2013-11-07T06:20:48,"Huh, anyway check out this you[tube] channel: ..."
1,adam riyati,2013-11-07T12:37:15,Hey guys check out my new channel and our firs...
2,Evgeny Murashkin,2013-11-08T17:34:21,just for test I have to say murdev.com
3,ElNino Melendez,2013-11-09T08:28:43,me shaking my sexy ass on my channel enjoy ^_^ ﻿
4,GsMega,2013-11-10T16:05:38,watch?v=vtaRGgvGtWQ Check this out .﻿


## Step 2 — Data Cleaning

In [13]:
# Check missing/null values across features
print("Missing values per feature:")
print(X.isnull().sum())

# Remove exact duplicate records while keeping X and y aligned
initial_records = len(X)
combined = pd.concat([X, y], axis=1)
cleaned = combined.drop_duplicates().reset_index(drop=True)
duplicates_dropped = initial_records - len(cleaned)

# Reassign cleaned features and target
X = cleaned[X.columns]
y = cleaned[y.columns]

print(f"\nInitial record count: {initial_records}")
print(f"Duplicates dropped: {duplicates_dropped}")
print(f"Remaining cleaned records: {len(X)}")

Missing values per feature:
AUTHOR       0
DATE       245
CONTENT      0
dtype: int64

Initial record count: 1956
Duplicates dropped: 16
Remaining cleaned records: 1940


## Step 3 — Data Splitting

`CLASS` is already a binary label (`0` = ham, `1` = spam), not a rating or score, so no threshold conversion is needed. The one-column target DataFrame is converted to a Series for Scikit-learn, then the data is split 80/20.

In [14]:
from sklearn.model_selection import train_test_split

# Convert the one-column target DataFrame to a Series
y = y.iloc[:, 0]

print("Target type: binary (0 = ham, 1 = spam)")
print("Target classes:", sorted(y.unique().tolist()))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

Target type: binary (0 = ham, 1 = spam)
Target classes: [0, 1]
Training set size: 1552
Testing set size: 388


## Step 4 — Model Training

The features are raw text, so a Random Forest cannot consume them directly. A small `Pipeline` first converts the `CONTENT` column (the comment text) into TF-IDF features, then feeds them to a `RandomForestClassifier`. `AUTHOR` and `DATE` are dropped: they are high-cardinality identifiers rather than meaningful signal. Because the vectorizer lives inside the pipeline, it is fitted on the training data only, which avoids leakage into the test set.

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# TF-IDF on the comment text only; all other columns are dropped
preprocessor = ColumnTransformer(
    [("text", TfidfVectorizer(lowercase=True, stop_words="english"), "CONTENT")],
    remainder="drop",
)

model = Pipeline([
    ("tfidf", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42)),
])

model.fit(X_train, y_train)

print("Random Forest training completed successfully.")

Random Forest training completed successfully.


## Step 5 — Model Evaluation

In [16]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Predict on the held-out test set
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=["ham (0)", "spam (1)"])

print(f"Accuracy: {accuracy * 100:.2f}%")
print("\nConfusion matrix (rows = actual, columns = predicted):")
print(cm)
print("\nClassification report:")
print(report)

Accuracy: 94.59%

Confusion matrix (rows = actual, columns = predicted):
[[185   8]
 [ 13 182]]

Classification report:
              precision    recall  f1-score   support

     ham (0)       0.93      0.96      0.95       193
    spam (1)       0.96      0.93      0.95       195

    accuracy                           0.95       388
   macro avg       0.95      0.95      0.95       388
weighted avg       0.95      0.95      0.95       388



## Step 6 — Sample Prediction / Inference

In [17]:
# Select one sample from the test set
sample = X_test.iloc[[0]]
actual_label = y_test.iloc[0]
predicted_label = model.predict(sample)[0]

label_names = {0: "ham", 1: "spam"}

print("Sample features:")
display(sample)
print("Actual label:   ", actual_label, f"({label_names[actual_label]})")
print("Predicted label:", predicted_label, f"({label_names[predicted_label]})")

Sample features:


,AUTHOR,DATE,CONTENT
1604,ovidiu mitricioiu,2015-05-26T08:11:50.918000,BEAUTIFUL


Actual label:    0 (ham)
Predicted label: 0 (ham)


In [22]:
# Take Author, Date and Comment from the user and predict its class
user_author = input("Enter Author name: ")
user_date = input("Enter Date (YYYY-MM-DD): ")
user_comment = input("Enter a YouTube comment: ")

# Create input in the same format used by the trained model
user_data = pd.DataFrame({
    "AUTHOR": [user_author],
    "DATE": [user_date],
    "CONTENT": [user_comment]
})

prediction = model.predict(user_data)[0]

print("\nInput Details:")
print("Author:", user_author)
print("Date:", user_date)
print("Comment:", user_comment)

if prediction == 1:
    print("\nOutput: SPAM")
else:
    print("\nOutput: HAM (Not Spam)")


Enter Author name:  dkjasdjahd
Enter Date (YYYY-MM-DD):  2025-09-25
Enter a YouTube comment:  check out my channel like



Input Details:
Author: dkjasdjahd
Date: 2025-09-25
Comment: check out my channel like

Output: SPAM
